# Darija Data Preparation Pipeline — GIZA++ + Thutmose + Benchmark


## STEP 0 — Install Dependencies (Kaggle-safe)

This version uses **real NVIDIA NeMo** for audio denoising/speech enhancement. If no NeMo enhancement checkpoint can be loaded, the denoising step stops with a clear error instead of silently using a spectral fallback.


In [1]:
import subprocess, sys, os, textwrap

DEPS_SENTINEL = '/kaggle/working/.darija_deps_installed_v2'

def pip_install(*pkgs, extra_args=None, name=None):
    args = [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs)
    if extra_args:
        args += extra_args
    result = subprocess.run(args, capture_output=True, text=True)
    label = name or ', '.join(pkgs)
    if result.returncode != 0:
        print(f'\nWARNING while installing {label}')
        print(result.stderr[-3000:])
    else:
        print(f'OK: {label}')
    return result.returncode == 0

def subprocess_import_check():
    code = """
import numpy, pandas, scipy, sklearn, pyarrow, datasets, transformers
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
print('pyarrow', pyarrow.__version__, 'has_json_', hasattr(pyarrow, 'json_'))
print('datasets', datasets.__version__)
print('transformers', transformers.__version__)
assert hasattr(pyarrow, 'json_'), 'pyarrow must provide json_ for the installed datasets package'
"""
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError('Dependency import check failed in a clean subprocess.')

if os.path.exists(DEPS_SENTINEL):
    print('Dependencies already installed in this Kaggle session. Running clean subprocess check...')
    subprocess_import_check()
    print('Dependency cell skipped. Continue with the next cell.')
else:
    print('Upgrading installer tools...')
    pip_install('pip', 'setuptools', 'wheel', extra_args=['--upgrade'], name='pip/setuptools/wheel')

    print('\nInstalling NVIDIA NeMo for Speech/Audio processing...')
    # Use the audio/asr/tts extras needed by AudioToAudioModel without relying on a half-installed audio collection.
    pip_install('nemo_toolkit[asr,tts,audio]', name='nemo_toolkit[asr,tts,audio]')

    print('\nInstalling audio packages...')
    pip_install('librosa==0.10.2.post1', 'soundfile', 'pydub', 'pyloudnorm', name='audio packages')

    print('\nInstalling NLP / training packages separately...')
    for pkg in ['transformers>=4.44', 'tokenizers', 'datasets>=3.0.0', 'accelerate', 'seqeval']:
        pip_install(pkg, name=pkg)

    print('\nInstalling Excel/data tools...')
    pip_install('openpyxl', 'xlsxwriter', 'tqdm', name='excel/data tools')

    print('\nFINAL repair of Kaggle binary scientific stack...')
    # Important: this is LAST because NeMo/librosa/datasets may change numpy/pyarrow again.
    # pyarrow>=19 is required because recent HuggingFace datasets imports pa.json_.
    pip_install(
        'numpy==1.26.4',
        'pandas==2.2.2',
        'scipy==1.14.1',
        'scikit-learn==1.5.2',
        'pyarrow==20.0.0',
        extra_args=['--force-reinstall', '--no-cache-dir'],
        name='FINAL numpy/pandas/scipy/sklearn/pyarrow compatible stack'
    )

    print('\nClean subprocess sanity check after final repair...')
    subprocess_import_check()

    with open(DEPS_SENTINEL, 'w') as f:
        f.write('ok')

    print('\nDependencies installed successfully.')
    print('IMPORTANT: Kaggle kernel must restart now so the current Python process drops old binary modules.')
    print('After restart, run the notebook again from the top. This cell will skip installation and continue.')

    os._exit(0)


Dependencies already installed in this Kaggle session. Running clean subprocess check...
numpy 1.26.4
pandas 2.2.2
scipy 1.14.1
sklearn 1.5.2
pyarrow 20.0.0 has_json_ True
datasets 4.8.3
transformers 4.57.6

Dependency cell skipped. Continue with the next cell.


In [2]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — enable GPU in Kaggle Settings > Accelerator')

Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


## STEP 0.5 — Load Files from Kaggle Dataset

In [3]:
import os, zipfile
import pandas as pd

BASE_DIR       = '/kaggle/working/darija_corpus'
AUDIO_RAW      = f'{BASE_DIR}/audio_raw'
AUDIO_STD      = f'{BASE_DIR}/audio_standard'
AUDIO_DENOISED = f'{BASE_DIR}/audio_denoised'
AUDIO_NORM     = f'{BASE_DIR}/audio_normalized'
OUTPUT_DIR     = f'{BASE_DIR}/outputs'
NER_MODEL_DIR  = f'{BASE_DIR}/ner_model'

for d in [AUDIO_RAW, AUDIO_STD, AUDIO_DENOISED, AUDIO_NORM, OUTPUT_DIR, NER_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Directories ready: {BASE_DIR}')

KAGGLE_INPUT = '/kaggle/input'
zip_found, excel_found, ner_zip_found = False, False, False
ZIP_PATH, EXCEL_PATH, NER_ZIP_PATH = '', '', ''

for root, dirs, files in os.walk(KAGGLE_INPUT):
    for fname in files:
        fp = os.path.join(root, fname)
        if fname.endswith('.zip') and 'ner_results' in fname.lower() and not ner_zip_found:
            NER_ZIP_PATH = fp; ner_zip_found = True; print(f'NER ZIP: {fp}')
        elif fname.endswith('.zip') and not zip_found:
            ZIP_PATH = fp; zip_found = True; print(f'Audio ZIP: {fp}')
        if fname == 'Results.xlsx' and not excel_found:
            EXCEL_PATH = fp; excel_found = True; print(f'Excel: {fp}')

# Manual overrides (uncomment if needed):
# ZIP_PATH     = '/kaggle/input/your-dataset/corpus.zip'
# EXCEL_PATH   = '/kaggle/input/your-dataset/Results.xlsx'
# NER_ZIP_PATH = '/kaggle/input/your-dataset/ner_results.zip'

if zip_found:
    with zipfile.ZipFile(ZIP_PATH) as z:
        for member in z.namelist():
            if member.endswith('.wav'):
                fname = os.path.basename(member)
                with z.open(member) as src, open(os.path.join(AUDIO_RAW, fname), 'wb') as dst:
                    dst.write(src.read())
    print('WAV files extracted')
else:
    import shutil
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for fname in files:
            if fname.endswith('.wav'):
                shutil.copy(os.path.join(root, fname), os.path.join(AUDIO_RAW, fname))

wav_count = len([f for f in os.listdir(AUDIO_RAW) if f.endswith('.wav')])
print(f'Total WAV: {wav_count}')

if not excel_found:
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for fname in files:
            if fname.endswith('.xlsx'):
                EXCEL_PATH = os.path.join(root, fname); excel_found = True; break

if excel_found:
    df_corpus = pd.read_excel(EXCEL_PATH)
    print(f'Excel loaded: {len(df_corpus)} rows | cols: {df_corpus.columns.tolist()}')
else:
    print('WARNING: No Excel found — using empty DataFrame')
    df_corpus = pd.DataFrame({'audio_id':[], 'ground_truth':[], 'qwen3-asr-checkpoint12600':[]})

Directories ready: /kaggle/working/darija_corpus
Excel: /kaggle/input/datasets/rajiamina/darija-speech-corpus/darija_corpus/Results.xlsx
Total WAV: 508
Excel loaded: 508 rows | cols: ['audio_id', 'ground_truth', 'gemini_transcription', 'qwen3-asr-checkpoint700', 'qwen3-asr-checkpoint3850', 'qwen3-asr-checkpoint12600', 'gemma-4-E2B-it', 'gemma-4-E4B-it']


## STEP 1 — Audio Standardization (16kHz mono 16-bit PCM)

In [4]:
import librosa, soundfile as sf, numpy as np
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

TARGET_SR = 16000

def standardize_audio(inp, out, sr=TARGET_SR):
    info = {'status':'ok','original_sr':None,'duration':None,'error':None}
    try:
        audio, orig_sr = librosa.load(inp, sr=None, mono=False)
        info['original_sr'] = orig_sr
        if audio.ndim > 1:
            audio = librosa.to_mono(audio)
        if orig_sr != sr:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
        info['duration'] = len(audio) / sr
        if info['duration'] < 0.1:
            info['status'] = 'too_short'; return info
        if np.sqrt(np.mean(audio**2)) < 1e-5:
            info['status'] = 'silent'; return info
        sf.write(out, (np.clip(audio,-1,1)*32767).astype(np.int16), sr, subtype='PCM_16')
    except Exception as e:
        info['status'] = 'error'; info['error'] = str(e)
    return info

wav_files = [f for f in os.listdir(AUDIO_RAW) if f.endswith('.wav')]
print(f'Files to standardize: {len(wav_files)}')
report = []
for f in tqdm(wav_files, desc='Standardizing'):
    r = standardize_audio(os.path.join(AUDIO_RAW,f), os.path.join(AUDIO_STD,f))
    r['file'] = f; report.append(r)
df_std = pd.DataFrame(report)
print(f"OK: {df_std[df_std.status=='ok'].shape[0]} | "
      f"silent: {df_std[df_std.status=='silent'].shape[0]} | "
      f"error: {df_std[df_std.status=='error'].shape[0]}")
if df_std['duration'].notna().any():
    print(df_std['duration'].describe().round(2))

Files to standardize: 508


Standardizing: 100%|██████████| 508/508 [00:18<00:00, 28.15it/s]

OK: 508 | silent: 0 | error: 0
count    508.00
mean      14.51
std        4.33
min        5.04
25%       11.44
50%       14.04
75%       17.00
max       28.98
Name: duration, dtype: float64


## STEP 2 — Audio Denoising with NVIDIA NeMo on a Small Sample

NeMo works, but it is slow on the full corpus. This step runs NeMo only on a small configurable sample, then copies the standardized audio for all remaining files into `AUDIO_DENOISED`. The rest of the pipeline can run on a complete audio directory without waiting hours.


In [5]:
import os, sys, types, shutil, torch
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm

# NeMo is correct but slow. Keep a small sample for testing/quality check.
NEMO_SAMPLE_SIZE = 10
REQUIRE_NEMO_DENOISING = True
DENOISE_METHOD = f"NeMo sample({NEMO_SAMPLE_SIZE}) + standardized copy for the rest"

NEMO_ENHANCEMENT_MODEL = os.environ.get(
    "NEMO_ENHANCEMENT_MODEL",
    "nvidia/se_den_sb_16k_small"
).strip()

nemo_enhancement_model = None
nemo_enhancement_name = None

def patch_nemo_runtime_compat():
    # Kaggle/Jupyter + datasets/pyarrow repeated registration.
    try:
        import pyarrow as pa
        for ext_name in [
            "datasets.features.features.Array2DExtensionType",
            "datasets.features.features.Array3DExtensionType",
            "datasets.features.features.Array4DExtensionType",
            "datasets.features.features.Array5DExtensionType",
        ]:
            try:
                pa.unregister_extension_type(ext_name)
            except Exception:
                pass
    except Exception:
        pass

    # Some NeMo checkpoints reference old audio loss paths and pass ndim.
    import nemo.core.classes.common as nemo_common

    class CompatibleMSELoss(torch.nn.Module):
        def __init__(self, reduction="mean", ndim=None, **kwargs):
            super().__init__()
            self.loss = torch.nn.MSELoss(reduction=reduction)
        def forward(self, estimate, target, **kwargs):
            return self.loss(estimate, target)

    class CompatibleMAELoss(torch.nn.Module):
        def __init__(self, reduction="mean", ndim=None, **kwargs):
            super().__init__()
            self.loss = torch.nn.L1Loss(reduction=reduction)
        def forward(self, estimate, target, **kwargs):
            return self.loss(estimate, target)

    for prefix in ["nemo.collections.common.losses", "nemo.collections.audio.losses"]:
        if prefix not in nemo_common.ALLOWED_NEMO_SUBMODULE_PREFIXES:
            nemo_common.ALLOWED_NEMO_SUBMODULE_PREFIXES.append(prefix)

    try:
        import nemo.collections.audio.losses as audio_losses
    except Exception:
        audio_losses = types.ModuleType("nemo.collections.audio.losses")
        sys.modules["nemo.collections.audio.losses"] = audio_losses

    audio_losses.MSELoss = CompatibleMSELoss
    audio_losses.MAELoss = CompatibleMAELoss

    for mod_name, cls_name, cls in [
        ("nemo.collections.audio.losses.MSELoss", "MSELoss", CompatibleMSELoss),
        ("nemo.collections.audio.losses.MAELoss", "MAELoss", CompatibleMAELoss),
        ("nemo.collections.common.losses.mse_loss", "MSELoss", CompatibleMSELoss),
        ("nemo.collections.common.losses.mae_loss", "MAELoss", CompatibleMAELoss),
    ]:
        mod = types.ModuleType(mod_name)
        setattr(mod, cls_name, cls)
        sys.modules[mod_name] = mod

def try_load_nemo_enhancement():
    global nemo_enhancement_model, nemo_enhancement_name
    patch_nemo_runtime_compat()
    from nemo.collections.audio.models.enhancement import SchroedingerBridgeAudioToAudioModel

    print(f"Trying NeMo model: {NEMO_ENHANCEMENT_MODEL}")
    if NEMO_ENHANCEMENT_MODEL.endswith(".nemo") or os.path.exists(NEMO_ENHANCEMENT_MODEL):
        model = SchroedingerBridgeAudioToAudioModel.restore_from(restore_path=NEMO_ENHANCEMENT_MODEL)
    else:
        model = SchroedingerBridgeAudioToAudioModel.from_pretrained(model_name=NEMO_ENHANCEMENT_MODEL)

    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    nemo_enhancement_model = model
    nemo_enhancement_name = NEMO_ENHANCEMENT_MODEL
    print(f"Loaded NeMo model: {nemo_enhancement_name}")
    return model

def _model_sample_rate(default_sr=16000):
    cfg = getattr(nemo_enhancement_model, "cfg", None)
    if cfg is not None:
        for key in ["sample_rate", "sr"]:
            try:
                sr = cfg.get(key)
                if sr:
                    return int(sr)
            except Exception:
                pass
    return default_sr

def _extract_audio_output(obj):
    if isinstance(obj, torch.Tensor):
        return obj
    if isinstance(obj, (list, tuple)):
        for item in obj:
            if isinstance(item, torch.Tensor):
                return item
    if isinstance(obj, dict):
        for key in ["output_signal", "enhanced_signal", "preds", "audio", "signal"]:
            if key in obj and isinstance(obj[key], torch.Tensor):
                return obj[key]
    raise RuntimeError(f"Unexpected NeMo output type: {type(obj)}")

def denoise_one_file(inp, out):
    model_sr = _model_sample_rate(default_sr=TARGET_SR)
    audio, _ = librosa.load(inp, sr=model_sr, mono=True)
    audio = audio.astype(np.float32)

    audio_3d = torch.from_numpy(audio).view(1, 1, -1)
    audio_2d = torch.from_numpy(audio).view(1, -1)
    length = torch.tensor([audio.shape[0]], dtype=torch.long)
    if torch.cuda.is_available():
        audio_3d = audio_3d.cuda()
        audio_2d = audio_2d.cuda()
        length = length.cuda()

    with torch.no_grad():
        try:
            enhanced = nemo_enhancement_model(input_signal=audio_3d, input_length=length)
        except Exception:
            enhanced = nemo_enhancement_model(input_signal=audio_2d, input_length=length)

    enhanced = _extract_audio_output(enhanced).detach().float().cpu().numpy().squeeze()
    peak = np.max(np.abs(enhanced)) if enhanced.size else 0
    if peak > 1.0:
        enhanced = enhanced / peak * 0.98
    sf.write(out, enhanced, model_sr)
    return {"status": "ok", "method": "nemo_audio", "model": nemo_enhancement_name, "output": out, "error": None}

std_files = sorted([f for f in os.listdir(AUDIO_STD) if f.endswith(".wav")])
nemo_files = std_files[:NEMO_SAMPLE_SIZE]
copy_files = std_files[NEMO_SAMPLE_SIZE:]

print(f"Total files: {len(std_files)}")
print(f"NeMo sample: {len(nemo_files)} | copied from standardized: {len(copy_files)}")

dr = []
if nemo_files:
    try_load_nemo_enhancement()
    for f in tqdm(nemo_files, desc=f"Denoising NeMo sample ({NEMO_SAMPLE_SIZE})"):
        inp = os.path.join(AUDIO_STD, f)
        out = os.path.join(AUDIO_DENOISED, f)
        try:
            r = denoise_one_file(inp, out)
        except Exception as e:
            r = {"status": "error", "method": "nemo_audio", "model": nemo_enhancement_name, "output": out, "error": str(e)}
            if REQUIRE_NEMO_DENOISING:
                print(f"NeMo denoising failed for {f}: {e}")
                raise
        r["file"] = f
        dr.append(r)

for f in tqdm(copy_files, desc="Copying standardized audio for remaining files"):
    src = os.path.join(AUDIO_STD, f)
    dst = os.path.join(AUDIO_DENOISED, f)
    shutil.copy(src, dst)
    dr.append({"file": f, "status": "ok", "method": "standardized_copy", "model": None, "output": dst, "error": None})

ok = sum(1 for r in dr if r["status"] == "ok")
print(f"Done: {ok}/{len(dr)} OK | Method: {DENOISE_METHOD}")
print(f"NeMo processed: {sum(1 for r in dr if r['method']=='nemo_audio')}")
print(f"Standardized copies: {sum(1 for r in dr if r['method']=='standardized_copy')}")


Total files: 508
NeMo sample: 10 | copied from standardized: 498


[NeMo W 2026-05-05 23:44:07 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Trying NeMo model: nvidia/se_den_sb_16k_small


se_den_sb_16k_small.nemo:   0%|          | 0.00/101M [00:00<?, ?B/s]

[NeMo W 2026-05-05 23:44:13 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /data/1618167/wsj0_enh_chime_train.json
    input_key: noisy_filepath
    target_key: clean_filepath
    audio_duration: 2.04
    random_offset: true
    normalize_input: true
    batch_size: 8
    shuffle: true
    num_workers: 40
    pin_memory: true
    
[NeMo W 2026-05-05 23:44:13 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/1618167/wsj0_enh_chime_val.json
    input_key: noisy_filepath
    target_key: clean_filepath
    normalize_input: false
    batch_size: 4
    shuffle: false
    num_worker

[NeMo I 2026-05-05 23:44:14 schroedinger_bridge:421] noise_schedule.time_max set to: 1.0
[NeMo I 2026-05-05 23:44:14 schroedinger_bridge:425] noise_schedule.time_min set to: 0.0001
[NeMo I 2026-05-05 23:44:14 schroedinger_bridge:428] noise_schedule.num_steps set to: 50
[NeMo I 2026-05-05 23:44:14 schroedinger_bridge:499] estimator time set to: previous
[NeMo I 2026-05-05 23:44:14 schroedinger_bridge:488] process set to: sde


[NeMo W 2026-05-05 23:44:14 enhancement:992] Metrics will be evaluated on first 50 examples of the evaluation datasets.


[NeMo I 2026-05-05 23:44:14 save_restore_connector:285] Model SchroedingerBridgeAudioToAudioModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--se_den_sb_16k_small/snapshots/fe9725b1ae90f7f6b11d9b95c5f9ccec736a211f/se_den_sb_16k_small.nemo.
Loaded NeMo model: nvidia/se_den_sb_16k_small


Copying standardized audio for remaining files: 100%|██████████| 498/498 [00:00<00:00, 2845.15it/s]

Done: 508/508 OK | Method: NeMo sample(10) + standardized copy for the rest
NeMo processed: 10
Standardized copies: 498


## STEP 3 — LUFS Audio Normalization (EBU R128 / NVIDIA standard)

In [6]:
import os
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm
import pyloudnorm as pyln

TARGET_LUFS    = -23.0
TARGET_PEAK_DB = -1.0

def simple_trim(audio, threshold=0.01):
    mask = np.abs(audio) > threshold
    if np.any(mask):
        start = np.argmax(mask)
        end = len(audio) - np.argmax(mask[::-1])
        return audio[start:end]
    return audio

def normalize_lufs(inp, out, tlufs=TARGET_LUFS, tpeak=TARGET_PEAK_DB):
    info = {'status':'ok','original_lufs':None,'gain_db':None,'error':None}
    try:
        data, sr = sf.read(inp)
        if data.dtype == np.int16:
            data = data.astype(np.float32) / 32768.0
        elif data.dtype != np.float32:
            data = data.astype(np.float32)
        if data.ndim > 1:
            data = data.mean(axis=1)
        meter = pyln.Meter(sr)
        loudness = meter.integrated_loudness(data)
        info['original_lufs'] = loudness
        if np.isinf(loudness) or loudness < -70:
            rms = np.sqrt(np.mean(data**2))
            if rms > 1e-8:
                data = data * (10**(tlufs/20) / rms)
        else:
            gain = 10**((tlufs - loudness)/20)
            data = data * gain
            info['gain_db'] = round(20*np.log10(gain), 2)
        peak = np.max(np.abs(data))
        peak_limit = 10**(tpeak/20)
        if peak > peak_limit:
            data = data * (peak_limit / peak)
        data = simple_trim(data)
        sf.write(out, (np.clip(data,-1,1)*32767).astype(np.int16), sr, subtype='PCM_16')
    except Exception as e:
        info['status'] = 'error'; info['error'] = str(e)
        shutil.copy(inp, out)
    return info

denoised_files = [f for f in os.listdir(AUDIO_DENOISED) if f.endswith('.wav')]
print(f'Files to normalize: {len(denoised_files)}')
nr = []
for f in tqdm(denoised_files, desc='LUFS Normalizing'):
    r = normalize_lufs(os.path.join(AUDIO_DENOISED,f), os.path.join(AUDIO_NORM,f))
    r['file'] = f; nr.append(r)
df_norm = pd.DataFrame(nr)
print(f"OK: {df_norm[df_norm.status=='ok'].shape[0]} | errors: {df_norm[df_norm.status=='error'].shape[0]}")
if df_norm['original_lufs'].notna().any():
    print(f"Mean original LUFS: {df_norm['original_lufs'].mean():.1f} dBFS")

Files to normalize: 508


LUFS Normalizing: 100%|██████████| 508/508 [00:08<00:00, 62.14it/s]

OK: 508 | errors: 0
Mean original LUFS: -21.9 dBFS


## STEP 3.5 — Audio Data Augmentation

This step creates optional augmented audio variants for robustness. It includes gain, noise, speed, pitch, time shift, polarity inversion, clipping, band-pass filtering, and light room/reverb simulation.

Keep `AUDIO_AUGMENTATION_ENABLED = False` if you only want to prepare clean normalized audio. Turn it on for training data generation.


In [7]:
# ============================================================
# STEP 3.5 — Audio augmentation variants
# ============================================================

import os, random, shutil
import numpy as np
import soundfile as sf
from scipy import signal
from tqdm import tqdm

AUDIO_AUGMENTATION_ENABLED = True
AUDIO_AUGMENTED = f'{BASE_DIR}/audio_augmented'
os.makedirs(AUDIO_AUGMENTED, exist_ok=True)

# Control dataset growth. Each normalized file can generate up to this many variants.
MAX_AUDIO_AUG_PER_FILE = 3
RANDOM_AUDIO_AUG_PER_FILE = 1
random.seed(SEED if 'SEED' in globals() else 42)
np.random.seed(SEED if 'SEED' in globals() else 42)

def _safe_audio(x):
    x = np.nan_to_num(x.astype(np.float32))
    peak = np.max(np.abs(x)) if len(x) else 0
    if peak > 1.0:
        x = x / peak * 0.98
    return np.clip(x, -1.0, 1.0)

def aug_gain(x, sr, db=None):
    db = random.uniform(-6, 6) if db is None else db
    return _safe_audio(x * (10 ** (db / 20)))

def aug_noise(x, sr, snr_db=None):
    snr_db = random.uniform(10, 30) if snr_db is None else snr_db
    noise = np.random.normal(0, 1, size=len(x)).astype(np.float32)
    sig_power = np.mean(x ** 2) + 1e-9
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = noise / (np.sqrt(np.mean(noise ** 2)) + 1e-9) * np.sqrt(noise_power)
    return _safe_audio(x + noise)

def aug_time_shift(x, sr, max_shift_ms=120):
    shift = int(random.uniform(-max_shift_ms, max_shift_ms) * sr / 1000)
    return _safe_audio(np.roll(x, shift))

def aug_speed(x, sr, rate=None):
    import librosa
    rate = random.uniform(0.9, 1.1) if rate is None else rate
    return _safe_audio(librosa.effects.time_stretch(x.astype(np.float32), rate=rate))

def aug_pitch(x, sr, steps=None):
    import librosa
    steps = random.uniform(-2, 2) if steps is None else steps
    return _safe_audio(librosa.effects.pitch_shift(x.astype(np.float32), sr=sr, n_steps=steps))

def aug_polarity(x, sr):
    return _safe_audio(-x)

def aug_clip(x, sr, clip_level=None):
    clip_level = random.uniform(0.65, 0.95) if clip_level is None else clip_level
    return _safe_audio(np.clip(x, -clip_level, clip_level) / clip_level * 0.98)

def aug_bandpass(x, sr, low=None, high=None):
    low = random.uniform(80, 250) if low is None else low
    high = random.uniform(3200, min(7600, sr / 2 - 100)) if high is None else high
    sos = signal.butter(4, [low, high], btype='band', fs=sr, output='sos')
    return _safe_audio(signal.sosfiltfilt(sos, x))

def aug_reverb(x, sr):
    delay_ms = random.uniform(25, 80)
    decay = random.uniform(0.15, 0.45)
    delay = max(1, int(delay_ms * sr / 1000))
    ir = np.zeros(delay * 4, dtype=np.float32)
    ir[0] = 1.0
    for k in range(1, 4):
        ir[k * delay] = decay ** k
    y = signal.fftconvolve(x, ir, mode='full')[:len(x)]
    return _safe_audio(y)

AUDIO_AUGMENTERS = {
    'gain': aug_gain,
    'noise': aug_noise,
    'shift': aug_time_shift,
    'speed': aug_speed,
    'pitch': aug_pitch,
    'polarity': aug_polarity,
    'clip': aug_clip,
    'bandpass': aug_bandpass,
    'reverb': aug_reverb,
}

def augment_audio_file(inp, out_dir, max_variants=MAX_AUDIO_AUG_PER_FILE, random_variants=RANDOM_AUDIO_AUG_PER_FILE):
    x, sr = sf.read(inp)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = x.astype(np.float32)
    if np.max(np.abs(x)) > 1.5:
        x = x / 32768.0
    base = os.path.splitext(os.path.basename(inp))[0]
    outputs = []

    selected = list(AUDIO_AUGMENTERS.items())[:max_variants]
    if random_variants:
        selected += random.sample(list(AUDIO_AUGMENTERS.items()), k=min(random_variants, len(AUDIO_AUGMENTERS)))

    seen = set()
    for name, fn in selected:
        suffix = name
        if suffix in seen:
            suffix = f'{name}_rand'
        seen.add(suffix)
        try:
            y = fn(x, sr)
            out = os.path.join(out_dir, f'{base}__aug_{suffix}.wav')
            sf.write(out, (np.clip(y, -1, 1) * 32767).astype(np.int16), sr, subtype='PCM_16')
            outputs.append(out)
        except Exception as e:
            print(f'Audio augmentation failed for {inp} / {name}: {e}')
    return outputs

if AUDIO_AUGMENTATION_ENABLED:
    norm_files = [f for f in os.listdir(AUDIO_NORM) if f.endswith('.wav')]
    audio_aug_report = []
    for f in tqdm(norm_files, desc='Audio augmentation'):
        outs = augment_audio_file(os.path.join(AUDIO_NORM, f), AUDIO_AUGMENTED)
        audio_aug_report.append({'file': f, 'augmentations': len(outs)})
    df_audio_aug = pd.DataFrame(audio_aug_report)
    print(f'Audio augmentation created {df_audio_aug["augmentations"].sum()} files in {AUDIO_AUGMENTED}')
else:
    print('Audio augmentation disabled. Set AUDIO_AUGMENTATION_ENABLED=True for training-data expansion.')


Audio augmentation disabled. Set AUDIO_AUGMENTATION_ENABLED=True for training-data expansion.


## STEP 4 — Token-Level Language Identification by Supervised Learning

This step trains a real token-classification model after balancing the training data:

- keep all French examples
- keep all English examples
- sample/oversample Darija and Arabic to the same reference size: `max(FRENCH_count, ENGLISH_count)`
- apply data augmentation only to the training split
- train XLM-RoBERTa with HuggingFace `Trainer`


In [8]:
# STEP 4a — Supervised Language ID learning after balancing
import os, re, unicodedata, warnings, random
import pandas as pd, numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

RUN_SUPERVISED_TRAINING = True
MODEL_NAME = "xlm-roberta-base"
LANG_MODEL_DIR = f"{BASE_DIR}/lang_id_model"
LOCAL_ITN_PARQUET = "/kaggle/input/data-ner-v2/itn_annotated_all.parquet"
LABELS = ["DARIJA", "ARABIC", "FRENCH", "ENGLISH"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
AR_RE = re.compile(r"[\u0600-\u06FF]")
LA_RE = re.compile(r"[A-Za-z]")

def normalize_label(x):
    x = str(x).strip().upper().replace("B-", "").replace("I-", "")
    aliases = {"FR": "FRENCH", "EN": "ENGLISH", "DA": "DARIJA", "DAR": "DARIJA", "AR": "ARABIC", "MSA": "ARABIC"}
    return aliases.get(x, x)

def find_parquet_files():
    files = []
    if os.path.exists(LOCAL_ITN_PARQUET):
        files.append(LOCAL_ITN_PARQUET)
    for root, _, names in os.walk("/kaggle/input"):
        for name in names:
            fp = os.path.join(root, name)
            if name.endswith(".parquet") and fp not in files:
                files.append(fp)
    return files

def load_lang_dataframe():
    dfs = []
    for fp in find_parquet_files():
        d = pd.read_parquet(fp)
        lower = {c.lower(): c for c in d.columns}
        word_col = next((lower[c] for c in ["word", "token", "tokens"] if c in lower), None)
        label_col = next((lower[c] for c in ["label", "lang", "language", "tag"] if c in lower), None)
        if word_col and label_col:
            dfs.append(d[[word_col, label_col]].rename(columns={word_col: "word", label_col: "label"}))
    if not dfs:
        raise FileNotFoundError("No usable parquet found. Need word/token column and label/lang column.")
    df = pd.concat(dfs, ignore_index=True).dropna()
    df["word"] = df["word"].astype(str).str.strip()
    df["label"] = df["label"].map(normalize_label)
    df = df[(df["word"] != "") & df["label"].isin(LABELS)]
    print("Raw label distribution:")
    print(df["label"].value_counts())
    return df

def balance_lang_data(df):
    """
    Exact requested balancing:
    - keep all French
    - keep all English
    - Darija and Arabic are sampled/oversampled to max(French, English)
    """
    counts = df["label"].value_counts().to_dict()
    fr_count = counts.get("FRENCH", 0)
    en_count = counts.get("ENGLISH", 0)
    target_size = max(fr_count, en_count)
    if target_size == 0:
        raise ValueError("FRENCH and ENGLISH are both empty, cannot use them as balancing reference.")

    parts = [df[df["label"] == "FRENCH"], df[df["label"] == "ENGLISH"]]
    for lab in ["DARIJA", "ARABIC"]:
        sub = df[df["label"] == lab]
        if len(sub) == 0:
            raise ValueError(f"No examples found for {lab}")
        replace = len(sub) < target_size
        parts.append(sub.sample(target_size, replace=replace, random_state=SEED))

    balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("\nBalanced distribution used for supervised learning:")
    print(balanced["label"].value_counts())
    return balanced

def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")

ARABIC_CHAR_VARIANTS = {"أ":["ا"],"إ":["ا"],"آ":["ا"],"ٱ":["ا"],"ى":["ي"],"ؤ":["و"],"ئ":["ي"],"ة":["ه","ت"],"گ":["ك"],"ڭ":["ك"],"پ":["ب"],"ڤ":["ف"]}
LATIN_KEYBOARD_NEIGHBORS = {"a":"qz","z":"as","e":"zr","r":"et","t":"ry","y":"tu","u":"yi","i":"uo","o":"ip","p":"o","q":"as","s":"qzd","d":"sf","f":"dg","g":"fh","h":"gj","j":"hk","k":"jl","l":"k","m":"n","n":"bm","b":"vn","v":"cb","c":"xv","x":"wc","w":"x"}
FR_EN_TRANSLIT_NOISE = {"ch":["sh"],"sh":["ch"],"ou":["u"],"oo":["ou"],"kh":["5"],"gh":["8"],"aa":["a"],"ee":["i"]}

def remove_random_char(word):
    if len(word) <= 3: return word
    i = random.randrange(len(word)); return word[:i] + word[i+1:]
def duplicate_random_char(word):
    if len(word) <= 2: return word
    i = random.randrange(len(word)); return word[:i] + word[i] + word[i:]
def swap_adjacent_chars(word):
    if len(word) <= 3: return word
    i = random.randrange(len(word)-1); chars=list(word); chars[i],chars[i+1]=chars[i+1],chars[i]; return "".join(chars)
def latin_keyboard_typo(word):
    chars=list(word); pos=[i for i,c in enumerate(chars) if c.lower() in LATIN_KEYBOARD_NEIGHBORS]
    if not pos: return word
    i=random.choice(pos); repl=random.choice(LATIN_KEYBOARD_NEIGHBORS[chars[i].lower()]); chars[i]=repl.upper() if chars[i].isupper() else repl
    return "".join(chars)
def arabic_shape_variant(word):
    variants=[word]
    for src_ch,repls in ARABIC_CHAR_VARIANTS.items():
        if src_ch in word:
            for repl in repls: variants.append(word.replace(src_ch,repl))
    return variants
def latin_translit_variant(word):
    variants={word}
    for src_pat,repls in FR_EN_TRANSLIT_NOISE.items():
        if src_pat in word.lower():
            for repl in repls: variants.add(re.sub(src_pat,repl,word,flags=re.I))
    return list(variants)
def augment_token(word, label):
    variants={word}
    if label in ["DARIJA","ARABIC"]:
        variants.update(arabic_shape_variant(word)); variants.add(word.replace("ـ",""))
    if label in ["FRENCH","ENGLISH"]:
        variants.update([word.lower(), word.upper(), word.capitalize(), strip_accents(word.lower())])
        variants.update(latin_translit_variant(word)); variants.add(latin_keyboard_typo(word))
    variants.update([remove_random_char(word), duplicate_random_char(word), swap_adjacent_chars(word)])
    return [v for v in variants if v and v.strip()]
def add_data_augmentation(df, factor=3):
    rows=[]
    for _, row in df.iterrows():
        variants=[v for v in augment_token(row["word"], row["label"]) if v != row["word"]]
        random.shuffle(variants)
        for v in variants[:factor]:
            rows.append({"word": v, "label": row["label"]})
    augmented = pd.concat([df, pd.DataFrame(rows)], ignore_index=True) if rows else df.copy()
    augmented = augmented.dropna()
    augmented["word"] = augmented["word"].astype(str).str.strip()
    augmented = augmented[augmented["word"] != ""].sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("\nTrain distribution after augmentation:")
    print(augmented["label"].value_counts())
    return augmented

def make_sequences(df, chunk_size=30):
    words, labs = df["word"].tolist(), df["label"].tolist()
    return {
        "tokens": [words[i:i+chunk_size] for i in range(0, len(words), chunk_size)],
        "ner_tags": [[label2id[l] for l in labs[i:i+chunk_size]] for i in range(0, len(labs), chunk_size)],
    }

def analyze_lang_rules(text):
    out=[]
    for i,tok in enumerate(str(text).split()):
        low=tok.lower()
        if AR_RE.search(tok):
            lang="DARIJA" if low in {"ديال","هاد","واش","بزاف","مزيان","ماشي","غادي","دابا","علاش","فين","كيفاش"} else "ARABIC"
        elif LA_RE.search(tok):
            lang="FRENCH" if re.search(r"[éèêàçùâîôû]|tion$|ment$|ique$", low) else "ENGLISH"
        else:
            lang="DARIJA"
        out.append({"token": tok, "lang": lang, "pos": i})
    return out

if RUN_SUPERVISED_TRAINING:
    df_lang_raw = load_lang_dataframe()
    df_lang_balanced = balance_lang_data(df_lang_raw)

    train_df, tmp_df = np.split(df_lang_balanced.sample(frac=1, random_state=SEED), [int(.8 * len(df_lang_balanced))])
    val_df, test_df = np.split(tmp_df.sample(frac=1, random_state=SEED), [int(.5 * len(tmp_df))])
    train_df = add_data_augmentation(train_df, factor=3)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    dataset = DatasetDict({
        "train": Dataset.from_dict(make_sequences(train_df)),
        "val": Dataset.from_dict(make_sequences(val_df)),
        "test": Dataset.from_dict(make_sequences(test_df)),
    })

    def tokenize_batch(examples):
        enc = tokenizer(examples["tokens"], is_split_into_words=True, padding="max_length", truncation=True, max_length=128)
        aligned = []
        for i, labels in enumerate(examples["ner_tags"]):
            prev = None; row = []
            for wid in enc.word_ids(batch_index=i):
                if wid is None: row.append(-100)
                elif wid != prev: row.append(labels[wid])
                else: row.append(-100)
                prev = wid
            aligned.append(row)
        enc["labels"] = aligned
        return enc

    dataset = dataset.map(tokenize_batch, batched=True)

    def compute_metrics(pred):
        logits, labels = pred
        preds = np.argmax(logits, axis=-1)
        yt, yp = [], []
        for pr, lb in zip(preds, labels):
            for p, l in zip(pr, lb):
                if l != -100:
                    yt.append(id2label[int(l)])
                    yp.append(id2label[int(p)])
        return {"accuracy": accuracy_score(yt, yp), "macro_f1": f1_score(yt, yp, average="macro")}

    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"{BASE_DIR}/lang_id_tmp",
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=25,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["val"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    supervised_eval = trainer.evaluate(dataset["test"])
    print("\nSupervised test metrics:", supervised_eval)
    print("\nClassification report on test:")
    pred = trainer.predict(dataset["test"])
    logits, labels = pred.predictions, pred.label_ids
    preds = np.argmax(logits, axis=-1)
    yt, yp = [], []
    for pr, lb in zip(preds, labels):
        for p, l in zip(pr, lb):
            if l != -100:
                yt.append(id2label[int(l)])
                yp.append(id2label[int(p)])
    print(classification_report(yt, yp, labels=LABELS, zero_division=0))

    os.makedirs(LANG_MODEL_DIR, exist_ok=True)
    trainer.save_model(LANG_MODEL_DIR)
    tokenizer.save_pretrained(LANG_MODEL_DIR)
    print(f"\nSaved supervised Language ID model to: {LANG_MODEL_DIR}")


2026-05-05 23:52:36.316090: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778025156.552987     931 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778025156.622056     931 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778025157.174792     931 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778025157.174828     931 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778025157.174830     931 computation_placer.cc:177] computation placer alr

Raw label distribution:
label
DARIJA     542595
ARABIC      88849
FRENCH       7790
ENGLISH      6341
Name: count, dtype: int64

Balanced distribution used for supervised learning:
label
ARABIC     7790
FRENCH     7790
DARIJA     7790
ENGLISH    6341
Name: count, dtype: int64

Train distribution after augmentation:
label
FRENCH     21836
ARABIC     21674
DARIJA     21038
ENGLISH    17015
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.879100,0.697985,0.702794,0.696879
2,0.748700,0.608358,0.747223,0.744917
3,0.704200,0.594153,0.754965,0.751828
4,0.653000,0.576121,0.755301,0.754055
5,0.641500,0.576642,0.754965,0.753424



Supervised test metrics: {'eval_loss': 0.5547614097595215, 'eval_accuracy': 0.7715343203230148, 'eval_macro_f1': 0.7703313662112089, 'eval_runtime': 1.1537, 'eval_samples_per_second': 86.678, 'eval_steps_per_second': 6.067, 'epoch': 5.0}

Classification report on test:
              precision    recall  f1-score   support

      DARIJA       0.74      0.61      0.67       776
      ARABIC       0.72      0.82      0.77       754
      FRENCH       0.80      0.86      0.83       786
     ENGLISH       0.83      0.79      0.81       656

    accuracy                           0.77      2972
   macro avg       0.77      0.77      0.77      2972
weighted avg       0.77      0.77      0.77      2972


Saved supervised Language ID model to: /kaggle/working/darija_corpus/lang_id_model


In [9]:
if os.path.exists(LANG_MODEL_DIR):
    print('Saved supervised Language ID model files:', os.listdir(LANG_MODEL_DIR)[:10])
else:
    print('No supervised model saved yet. Run STEP 4a first.')


Saved supervised Language ID model files: ['special_tokens_map.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors', 'sentencepiece.bpe.model', 'tokenizer.json', 'config.json']


In [10]:
# STEP 4c — unified Language ID API
import torch, torch.nn.functional as F
from transformers import AutoModelForTokenClassification, AutoTokenizer
USE_SUPERVISED_LANG_ID = os.path.exists(LANG_MODEL_DIR)
if USE_SUPERVISED_LANG_ID:
    print('Using supervised XLM-R Language ID model.')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(LANG_MODEL_DIR)
    lang_model = AutoModelForTokenClassification.from_pretrained(LANG_MODEL_DIR).to(device).eval()
    def analyze_lang_supervised(text):
        if not isinstance(text, str) or not text.strip(): return []
        words = text.split()
        enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad(): probs = F.softmax(lang_model(**{k:v.to(device) for k,v in enc.items()}).logits, dim=-1)
        out=[]; prev=None
        for i, wid in enumerate(enc.word_ids()):
            if wid is None or wid == prev: continue
            pred = torch.argmax(probs[0, i]).item()
            out.append({'token':words[wid], 'lang':id2label[pred], 'pos':wid}); prev = wid
        return out
else:
    print('Supervised model not found. Falling back to deterministic rule labels.')
    analyze_lang_supervised = analyze_lang_rules

def analyze_lang(text, method='supervised'):
    return analyze_lang_rules(text) if method == 'rules' else analyze_lang_supervised(text)

def lang_dist(text, method='supervised'):
    items = analyze_lang(text, method=method); total = len(items) or 1; counts = {}
    for it in items: counts[it['lang']] = counts.get(it['lang'], 0) + 1
    return {k: round(100*v/total, 1) for k,v in counts.items()}

for txt in ['هاد الحلوة وما محتاجاش pommade', 'this test فيه العربية و français']:
    print('\nTEXT:', txt); print('RULES:', analyze_lang(txt, method='rules')); print('SUPERVISED:', analyze_lang(txt))


Using supervised XLM-R Language ID model.


The tokenizer you are loading from '/kaggle/working/darija_corpus/lang_id_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



TEXT: هاد الحلوة وما محتاجاش pommade
RULES: [{'token': 'هاد', 'lang': 'DARIJA', 'pos': 0}, {'token': 'الحلوة', 'lang': 'ARABIC', 'pos': 1}, {'token': 'وما', 'lang': 'ARABIC', 'pos': 2}, {'token': 'محتاجاش', 'lang': 'ARABIC', 'pos': 3}, {'token': 'pommade', 'lang': 'ENGLISH', 'pos': 4}]
SUPERVISED: [{'token': 'هاد', 'lang': 'DARIJA', 'pos': 0}, {'token': 'الحلوة', 'lang': 'ARABIC', 'pos': 1}, {'token': 'وما', 'lang': 'DARIJA', 'pos': 2}, {'token': 'محتاجاش', 'lang': 'DARIJA', 'pos': 3}, {'token': 'pommade', 'lang': 'FRENCH', 'pos': 4}]

TEXT: this test فيه العربية و français
RULES: [{'token': 'this', 'lang': 'ENGLISH', 'pos': 0}, {'token': 'test', 'lang': 'ENGLISH', 'pos': 1}, {'token': 'فيه', 'lang': 'ARABIC', 'pos': 2}, {'token': 'العربية', 'lang': 'ARABIC', 'pos': 3}, {'token': 'و', 'lang': 'ARABIC', 'pos': 4}, {'token': 'français', 'lang': 'FRENCH', 'pos': 5}]
SUPERVISED: [{'token': 'this', 'lang': 'DARIJA', 'pos': 0}, {'token': 'test', 'lang': 'ARABIC', 'pos': 1}, {'token': 'فيه',

## STEP 4d — Benchmark Rule Labels vs Supervised Learning

This benchmark uses the held-out parquet split when available. It compares the first-pass deterministic labeling method with the supervised model trained from the parquet data.


In [11]:
benchmark_lang_id = pd.DataFrame()
if 'test_df' in globals() and len(test_df):
    eval_df = test_df.copy().reset_index(drop=True)
    y_true = eval_df['label'].tolist()
    y_rules = [analyze_lang_rules(w)[0]['lang'] if analyze_lang_rules(w) else 'DARIJA' for w in eval_df['word']]
    y_sup = [analyze_lang_supervised(w)[0]['lang'] if analyze_lang_supervised(w) else 'DARIJA' for w in eval_df['word']]
    rows = []
    for name, pred in [('rules_first_pass', y_rules), ('supervised_xlm_roberta', y_sup)]:
        rows.append({'method':name, 'accuracy':accuracy_score(y_true, pred), 'macro_f1':f1_score(y_true, pred, labels=LABELS, average='macro'), 'weighted_f1':f1_score(y_true, pred, labels=LABELS, average='weighted')})
        print('\n', name); print(classification_report(y_true, pred, labels=LABELS, zero_division=0))
    benchmark_lang_id = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
    display(benchmark_lang_id)
else:
    print('No held-out parquet test split is available. Run STEP 4a with RUN_SUPERVISED_TRAINING=True.')



 rules_first_pass
              precision    recall  f1-score   support

      DARIJA       0.91      0.07      0.12       776
      ARABIC       0.26      1.00      0.41       754
      FRENCH       0.00      0.00      0.00       786
     ENGLISH       0.00      0.00      0.00       656

    accuracy                           0.27      2972
   macro avg       0.29      0.27      0.13      2972
weighted avg       0.30      0.27      0.14      2972


 supervised_xlm_roberta
              precision    recall  f1-score   support

      DARIJA       0.67      0.50      0.57       776
      ARABIC       0.66      0.87      0.75       754
      FRENCH       0.68      0.84      0.75       786
     ENGLISH       0.84      0.56      0.68       656

    accuracy                           0.70      2972
   macro avg       0.71      0.69      0.69      2972
weighted avg       0.71      0.70      0.69      2972



,method,accuracy,macro_f1,weighted_f1
1,supervised_xlm_roberta,0.696501,0.687292,0.687520
0,rules_first_pass,0.270525,0.133693,0.136596


## STEP 5 — Text Normalization + Semiotic Labeling Setup

This section keeps the Darija spoken-to-written normalization, with comments explaining the UTF-8 Arabic mappings. Semiotic label definitions are handled in STEP 7 by two methods:

1. **Method 1:** GIZA++ alignment + Thutmose-style rule tagger.
2. **Method 2:** supervised learning from `itn_annotated_all.parquet`.


In [ ]:
import re

# Arabic UTF-8 normalization maps:
# ALIF_V: normalize Arabic alif variants (أ إ آ ٱ) to bare alif (ا)
# HAMZA: normalize hamza-on-waw/ya (ؤ ئ) to waw/ya (و ي)
# ALIF_MQ: normalize alif maqsura (ى) to ya (ي)
# DARI: normalize Moroccan/Persian letters often seen in Darija text:
#       gaf/keheh/peh (گ ک پ) to Arabic kaf/ba (ك ك ب)
ALIF_V  = {'أ':'ا','إ':'ا','آ':'ا','ٱ':'ا'}
HAMZA   = {'ؤ':'و','ئ':'ي'}
ALIF_MQ = {'ى':'ي'}
DARI    = {'گ':'ك','ک':'ك','پ':'ب'}

# TSHKL: Arabic diacritics. TATWL: tatweel/kashida. DIRNL: invisible bidi/control chars.
TSHKL = set('ًٌٍَُِّْٰٕٓٔ')
TATWL = 'ـ'
DIRNL = {'‏','‎','‫','‪','‬','​','﻿','⁧','⁦'}

# Darija number words used for inverse text normalization.
NUMS = {
    0:'صفر', 1:'واحد', 2:'جوج', 3:'تلاتة', 4:'ربعة', 5:'خمسة',
    6:'ستة', 7:'سبعة', 8:'تمنية', 9:'تسعود', 10:'عشرة',
    11:'حضاش', 12:'تناش', 20:'عشرين', 30:'تلاتين', 40:'ربعين',
    50:'خمسين', 60:'ستين', 70:'سبعين', 80:'تمانين', 90:'تسعين',
    100:'مية', 200:'ميتين', 1000:'الف', 2000:'جوج الالاف',
    1000000:'مليون',
}

def n2da(n):
    if n in NUMS: return NUMS[n]
    if n < 0: return 'ناقص ' + n2da(-n)
    if n >= 1000000:
        m, r = divmod(n, 1000000)
        return n2da(m) + ' مليون' + (' و' + n2da(r) if r else '')
    if n >= 1000:
        t, r = divmod(n, 1000)
        b = {1:'الف', 2:'جوج الالاف'}.get(t, n2da(t) + ' الالاف')
        return b + (' و' + n2da(r) if r else '')
    if n >= 100:
        h, r = divmod(n, 100)
        b = NUMS.get(h * 100, str(h * 100))
        return b + (' و' + n2da(r) if r else '')
    if n >= 21:
        t, o = divmod(n, 10)
        return NUMS.get(o, str(o)) + ' و' + NUMS.get(t * 10, str(t * 10))
    return str(n)

MONTHS = {
    1:'يناير', 2:'فبراير', 3:'مارس', 4:'ابريل', 5:'ماي', 6:'يونيو',
    7:'يوليوز', 8:'غشت', 9:'شتنبر', 10:'اكتوبر', 11:'نونبر', 12:'دجنبر'
}

UNITS = {
    'kg':'كيلو', 'g':'غرام', 'km':'كيلومتر', 'm':'متر',
    'cm':'سنتيمتر', 'mm':'ميليمتر', 'l':'ليتر', 'ml':'ميليلتر',
    'cl':'سنتيليتر', 'dh':'درهم', 'mad':'درهم', 'dhs':'دراهم'
}

def rm_dir(text):
    for c in DIRNL:
        text = text.replace(c, '')
    return text

def norm_letters(text):
    for s, t in {**ALIF_V, **HAMZA, **ALIF_MQ, **DARI}.items():
        text = text.replace(s, t)
    text = ''.join(c for c in text if c not in TSHKL)
    return text.replace(TATWL, '')

def norm_nums(text):
    # Arabic-Indic digits ٠١٢٣٤٥٦٧٨٩ -> European digits 0123456789
    hindi_digits = '٠١٢٣٤٥٦٧٨٩'
    text = text.translate(str.maketrans(hindi_digits, '0123456789'))
    def decimal_repl(m):
        return n2da(int(m.group(1))) + ' فاصلة ' + ' '.join(n2da(int(d)) for d in m.group(2))
    text = re.sub(r'(\d+)[.,](\d+)', decimal_repl, text)
    def int_repl(m):
        n = int(m.group())
        return n2da(n) if n <= 10000000 else m.group()
    return re.sub(r'\b\d+\b', int_repl, text)

def norm_dates(text):
    def repl(m):
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return f'{n2da(d)} {MONTHS.get(mo, str(mo))} {n2da(y)}'
    return re.sub(r'(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})', repl, text)

def norm_time(text):
    def repl(m):
        h, mn = int(m.group(1)), int(m.group(2))
        hs = n2da(h)
        if mn == 0: return hs + ' دالعشية'
        if mn == 30: return hs + ' ونص'
        if mn == 15: return hs + ' وربع'
        if mn == 45: return hs + ' الا ربع'
        return hs + ' و' + n2da(mn) + ' دقائق'
    return re.sub(r'(\d{1,2})[h:](\d{2})', repl, text)

def norm_units(text):
    def repl(m):
        n = m.group(1)
        u = m.group(2).lower()
        return n2da(int(n)) + ' ' + UNITS.get(u, u) if n.isdigit() else m.group()
    pats = '|'.join(re.escape(k) for k in sorted(UNITS, key=len, reverse=True))
    return re.sub(r'(\d+)\s*(' + pats + r')', repl, text, flags=re.I)

def spoken_to_written(text, normalize_numbers=True):
    if not isinstance(text, str): return text
    t = rm_dir(text.strip())
    t = norm_letters(t)
    t = norm_dates(t)
    t = norm_time(t)
    t = norm_units(t)
    if normalize_numbers:
        t = norm_nums(t)
    return re.sub(r'\s+', ' ', t).strip()

print('Text normalization ready with commented UTF-8 Arabic mappings.')


## STEP 6 — Real GIZA++ Alignment

This replaces the previous fast_align implementation. GIZA++ is cloned/compiled and used through plain2snt.out, snt2cooc.out, and GIZA++.


In [13]:
import os, subprocess, re
from pathlib import Path
GIZA_ROOT = Path('/kaggle/working/giza-pp')
GIZA_BIN = GIZA_ROOT / 'GIZA++-v2' / 'GIZA++'
PLAIN2SNT = GIZA_ROOT / 'GIZA++-v2' / 'plain2snt.out'
SNT2COOC = GIZA_ROOT / 'GIZA++-v2' / 'snt2cooc.out'
GIZA_ALIGN_TABLE = {}

def install_giza_pp():
    if GIZA_BIN.exists() and PLAIN2SNT.exists() and SNT2COOC.exists(): return True
    if not GIZA_ROOT.exists(): subprocess.run(['git','clone','https://github.com/moses-smt/giza-pp.git',str(GIZA_ROOT)], check=True)
    subprocess.run(['make'], cwd=GIZA_ROOT, check=True)
    return GIZA_BIN.exists() and PLAIN2SNT.exists() and SNT2COOC.exists()

def tokenize_for_giza(text):
    return ' '.join(re.findall(r'[\u0600-\u06FFA-Za-z]+|\d|[^\s]', str(text).strip()))

def parse_a3_final(a3_path):
    alignments=[]; lines=Path(a3_path).read_text(encoding='utf-8', errors='ignore').splitlines()
    for i in range(0, len(lines), 3):
        if i+2 >= len(lines): continue
        src_items = re.findall(r'(\S+) \(\{ ([^}]*) \}\)', lines[i+2])
        pairs=[]; si=-1
        for src_tok, js in src_items:
            if src_tok == 'NULL': continue
            si += 1
            for j in js.split():
                if j.isdigit(): pairs.append((si, int(j)-1))
        alignments.append(pairs)
    return alignments

def fallback_align(spoken_toks, written_toks):
    return [{'si':i,'wi':i if i < len(written_toks) else None,'st':s,'wt':written_toks[i] if i < len(written_toks) else None,'type':'fallback_length','sim':0.0} for i,s in enumerate(spoken_toks)]

def build_giza_alignment_cache(pairs, work_dir=f'{BASE_DIR}/giza_work'):
    os.makedirs(work_dir, exist_ok=True)
    if not install_giza_pp(): raise RuntimeError('GIZA++ binaries are not available.')
    src_path=Path(work_dir)/'spoken.txt'; trg_path=Path(work_dir)/'written.txt'
    src_path.write_text('\n'.join(tokenize_for_giza(s) for s,_ in pairs)+'\n', encoding='utf-8')
    trg_path.write_text('\n'.join(tokenize_for_giza(t) for _,t in pairs)+'\n', encoding='utf-8')
    subprocess.run([str(PLAIN2SNT), str(src_path), str(trg_path)], cwd=work_dir, check=True)
    src_vcb=Path(work_dir)/'spoken.txt.vcb'; trg_vcb=Path(work_dir)/'written.txt.vcb'; snt=Path(work_dir)/'spoken.txt_written.txt.snt'; cooc=Path(work_dir)/'spoken_written.cooc'
    with open(cooc,'w',encoding='utf-8') as fh: subprocess.run([str(SNT2COOC), str(src_vcb), str(trg_vcb), str(snt)], cwd=work_dir, check=True, stdout=fh)
    out_prefix=Path(work_dir)/'giza_itn'
    subprocess.run([str(GIZA_BIN), '-S', str(src_vcb), '-T', str(trg_vcb), '-C', str(snt), '-CoocurrenceFile', str(cooc), '-outputpath', str(out_prefix)], cwd=work_dir, check=True)
    parsed=parse_a3_final(str(out_prefix)+'.A3.final'); GIZA_ALIGN_TABLE.clear()
    for (spoken,written), pairs_idx in zip(pairs, parsed):
        stoks=tokenize_for_giza(spoken).split(); wtoks=tokenize_for_giza(written).split()
        GIZA_ALIGN_TABLE[(spoken,written)] = [{'si':si,'wi':wi,'st':stoks[si] if si < len(stoks) else None,'wt':wtoks[wi] if wi < len(wtoks) else None,'type':'giza++','sim':1.0} for si,wi in pairs_idx]
    print(f'GIZA++ alignment cache built: {len(GIZA_ALIGN_TABLE)} sentence pairs')

def align_tokens(spoken_toks, written_toks, spoken_text=None, written_text=None):
    if spoken_text is not None and written_text is not None and (spoken_text, written_text) in GIZA_ALIGN_TABLE: return GIZA_ALIGN_TABLE[(spoken_text, written_text)]
    return fallback_align(spoken_toks, written_toks)

if 'df_corpus' in globals() and len(df_corpus):
    cols=df_corpus.columns.tolist(); gt_col='ground_truth' if 'ground_truth' in cols else (cols[1] if len(cols)>1 else cols[0])
    sample_pairs=[]
    for x in df_corpus[gt_col].dropna().astype(str).head(2000):
        sp=re.sub(r'\s+',' ', rm_dir(x.strip())).strip(); wr=spoken_to_written(sp, normalize_numbers=True)
        if sp and wr: sample_pairs.append((sp,wr))
    try: build_giza_alignment_cache(sample_pairs)
    except Exception as e: print('GIZA++ could not run:', e)


Cloning into '/kaggle/working/giza-pp'...


make -C GIZA++-v2
make[1]: Entering directory '/kaggle/working/giza-pp/GIZA++-v2'
mkdir optimized/
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c Parameter.cpp -o optimized/Parameter.o


Parameter.cpp: In function ‘bool writeParameters(std::ofstream&, const ParSet&, int)’:
Parameter.cpp:48:21: warning: ignoring return value of ‘char* getcwd(char*, size_t)’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
   48 |               getcwd(path,1024);
      |               ~~~~~~^~~~~~~~~~~


g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c myassert.cpp -o optimized/myassert.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c Perplexity.cpp -o optimized/Perplexity.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c model1.cpp -o optimized/model1.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c model2.cpp -o optimized/model2.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c model3.cpp -o optimized/model3.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c getSentence.cpp -o optimized/getSentence.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE 

w1:/kaggle/working/darija_corpus/giza_work/spoken w2:/kaggle/working/darija_corpus/giza_work/written
Vocabulary does not exist.
Vocabulary does not exist.
END.


In [ ]:
# STEP 7 — Two semiotic-label methods + benchmarking
import os, re, random, warnings
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

warnings.filterwarnings("ignore")

SEMIOTIC_LABELS = [
    "CARDINAL", "MONEY", "DATE", "ORDINAL", "MEASURE", "DECIMAL",
    "LETTER", "TIME", "PUNCT", "TELEPHONE", "ADDRESS", "ELECTRONIC"
]
semiotic_label2id = {l: i for i, l in enumerate(SEMIOTIC_LABELS)}
semiotic_id2label = {i: l for l, i in semiotic_label2id.items()}
ITN_MODEL_DIR = f"{BASE_DIR}/itn_semiotic_model"
LOCAL_ITN_PARQUET = "/kaggle/input/data-ner-v2/itn_annotated_all.parquet"
RUN_ITN_SUPERVISED_TRAINING = True
ITN_MODEL_NAME = "xlm-roberta-base"

class ThutmoseRuleTagger:
    """
    Method 1: Thutmose-style rule tagger.
    It covers the required semiotic classes:
    CARDINAL, MONEY, DATE, ORDINAL, MEASURE, DECIMAL, LETTER, TIME,
    PUNCT, TELEPHONE, ADDRESS, ELECTRONIC.
    """
    ORDINAL_WORDS = {"اول", "الأول", "الاول", "ثاني", "الثاني", "تالت", "الثالث", "رابع", "خامس"}
    DECIMAL_WORDS = {"نص", "نصف", "نصين", "ربع", "ثلث"}
    ADDRESS_HINTS = {"شارع", "زنقة", "حي", "درب", "اقامة", "إقامة", "عمارة", "الساماني"}

    PATTERNS = [
        ("ELECTRONIC", re.compile(r"^[\w.\-]+@[\w.\-]+\.\w+$|^(https?://|www\.)\S+$", re.I)),
        ("TELEPHONE",  re.compile(r"^\+?\d[\d\s.\-]{5,}\d$")),
        ("DATE",       re.compile(r"^(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}|\d{4}[/\-]\d{1,2}[/\-]\d{1,2}|\d{4})$")),
        ("TIME",       re.compile(r"^\d{1,2}[h:]\d{2}(\s*(am|pm))?$|^\d+\s*(ساعة|ساعات|دقيقة|دقائق)$", re.I)),
        ("MONEY",      re.compile(r"^\d+([,.]\d+)?\s*(dh|mad|dhs|درهم|دراهم)$", re.I)),
        ("MEASURE",    re.compile(r"^\d+([,.]\d+)?\s*(kg|g|mg|km|m|cm|mm|l|ml|cl|سنة|سنين|عام|اعوام|متر|كيلو)$", re.I)),
        ("DECIMAL",    re.compile(r"^-?\d+[,.]\d+$")),
        ("ORDINAL",    re.compile(r"^\d+(eme|er|ere|st|nd|rd|th)$", re.I)),
        ("CARDINAL",   re.compile(r"^[٠-٩\d]+$")),
        ("LETTER",     re.compile(r"^([A-Za-z]|[A-Za-z]{2,}|[أ-ي]\.?){1,6}$")),
        ("PUNCT",      re.compile(r"^[،؛؟!?,.:;()\[\]\"_+*/|<>&@#\-]+$")),
    ]

    def tag_span(self, span, context=""):
        s = str(span).strip()
        if not s:
            return "O"
        norm = norm_letters(s)
        if norm in self.ORDINAL_WORDS:
            return "ORDINAL"
        if norm in self.DECIMAL_WORDS:
            return "DECIMAL"
        if any(h in norm for h in self.ADDRESS_HINTS):
            return "ADDRESS"
        for cls, pat in self.PATTERNS:
            if pat.match(s):
                return cls
        if re.search(r"\d", s) and re.search(r"(درهم|دراهم|dh|mad|dhs)", s, flags=re.I):
            return "MONEY"
        if re.search(r"\d", s) and re.search(r"(سنة|سنين|عام|متر|كيلو|kg|cm|km|ml)", s, flags=re.I):
            return "MEASURE"
        return "O"

    def extract_candidates(self, text):
        if not isinstance(text, str):
            return []
        patterns = [
            r"\+?\d[\d\s.\-]{5,}\d",
            r"\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}",
            r"\d{1,2}[h:]\d{2}",
            r"\d+([,.]\d+)?\s*(dh|mad|dhs|درهم|دراهم)",
            r"\d+([,.]\d+)?\s*(kg|g|mg|km|m|cm|mm|l|ml|cl|سنة|سنين|عام|متر|كيلو)",
            r"[\w.\-]+@[\w.\-]+\.\w+|https?://\S+|www\.\S+",
            r"[٠-٩\d]+([,.][٠-٩\d]+)?",
            r"[،؛؟!?,.:;()\[\]\"_+*/|<>&@#\-]+",
        ]
        spans = []
        for pat in patterns:
            for m in re.finditer(pat, text, flags=re.I):
                spans.append((m.start(), m.end(), m.group()))
        for w in str(text).split():
            if norm_letters(w) in self.ORDINAL_WORDS or norm_letters(w) in self.DECIMAL_WORDS:
                start = text.find(w)
                spans.append((start, start + len(w), w))
        spans = sorted(set(spans))
        return [{"start": s, "end": e, "token": tok, "class": self.tag_span(tok, text)} for s, e, tok in spans]

    def tag_text(self, text):
        return self.extract_candidates(text)

    def summary(self, text):
        out = {}
        for item in self.tag_text(text):
            cls = item["class"]
            if cls != "O":
                out[cls] = out.get(cls, 0) + 1
        return out

tagger_heuristic = ThutmoseRuleTagger()

def find_itn_parquet_files():
    files = []
    if os.path.exists(LOCAL_ITN_PARQUET):
        files.append(LOCAL_ITN_PARQUET)
    for root, _, names in os.walk("/kaggle/input"):
        for name in names:
            fp = os.path.join(root, name)
            if name.endswith(".parquet") and fp not in files:
                files.append(fp)
    return files

def load_itn_dataframe():
    dfs = []
    for fp in find_itn_parquet_files():
        d = pd.read_parquet(fp)
        lower = {c.lower(): c for c in d.columns}
        sent_col = lower.get("sentence") or lower.get("text")
        span_col = lower.get("span") or lower.get("token") or lower.get("word")
        label_col = lower.get("label") or lower.get("semiotic_class") or lower.get("class")
        if sent_col and span_col and label_col:
            dfs.append(d[[sent_col, span_col, label_col]].rename(columns={sent_col:"sentence", span_col:"span", label_col:"label"}))
    if not dfs:
        raise FileNotFoundError("No ITN parquet with sentence/span/label columns was found.")
    df = pd.concat(dfs, ignore_index=True).dropna()
    df["sentence"] = df["sentence"].astype(str)
    df["span"] = df["span"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    df = df[(df["span"] != "") & df["label"].isin(SEMIOTIC_LABELS)]
    print("ITN raw distribution:")
    print(df["label"].value_counts())
    return df

def balance_itn_training_data(df, target_per_class=3000):
    parts = []
    for lab in SEMIOTIC_LABELS:
        sub = df[df["label"] == lab]
        if len(sub) == 0:
            continue
        take = min(target_per_class, max(target_per_class, len(sub)))
        parts.append(sub.sample(target_per_class, replace=len(sub) < target_per_class, random_state=SEED))
    balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("\nITN balanced training distribution:")
    print(balanced["label"].value_counts())
    return balanced

def augment_itn_span(span, label):
    variants = {span}
    s = str(span)
    if label in {"CARDINAL", "DATE", "MONEY", "MEASURE", "TIME", "TELEPHONE", "DECIMAL"}:
        variants.add(s.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")))
        variants.add(s.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")))
    if label in {"MONEY"}:
        variants.update([re.sub(r"(درهم|دراهم|dh|mad|dhs)", "MAD", s, flags=re.I), re.sub(r"(درهم|دراهم|dh|mad|dhs)", "درهم", s, flags=re.I)])
    if label in {"MEASURE"}:
        variants.update([s.replace("سنة", "عام"), s.replace("kg", "كيلو")])
    variants.add(norm_letters(s))
    return [v for v in variants if v and v.strip()]

def add_itn_data_augmentation(df, factor=2):
    rows = []
    for _, row in df.iterrows():
        variants = [v for v in augment_itn_span(row["span"], row["label"]) if v != row["span"]]
        random.shuffle(variants)
        for v in variants[:factor]:
            rows.append({"sentence": row["sentence"], "span": v, "label": row["label"]})
    out = pd.concat([df, pd.DataFrame(rows)], ignore_index=True) if rows else df.copy()
    out = out.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("\nITN train distribution after augmentation:")
    print(out["label"].value_counts())
    return out

def make_itn_input(row):
    return f"span: {row['span']} </s> sentence: {row['sentence']}"

def train_itn_supervised_model():
    global itn_tokenizer, itn_model, itn_trainer, itn_test_df, benchmark_itn
    df = load_itn_dataframe()
    train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["label"])
    train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label"])

    train_bal = balance_itn_training_data(train_df, target_per_class=3000)
    train_bal = add_itn_data_augmentation(train_bal, factor=2)

    for frame in [train_bal, val_df, test_df]:
        frame["text"] = frame.apply(make_itn_input, axis=1)
        frame["labels"] = frame["label"].map(semiotic_label2id)

    itn_tokenizer = AutoTokenizer.from_pretrained(ITN_MODEL_NAME)
    ds = DatasetDict({
        "train": Dataset.from_pandas(train_bal[["text", "labels"]], preserve_index=False),
        "val": Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False),
        "test": Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False),
    })

    def tok(batch):
        return itn_tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

    ds = ds.map(tok, batched=True)
    itn_model = AutoModelForSequenceClassification.from_pretrained(
        ITN_MODEL_NAME,
        num_labels=len(SEMIOTIC_LABELS),
        id2label=semiotic_id2label,
        label2id=semiotic_label2id,
    )

    def metrics(pred):
        logits, labels = pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

    args = TrainingArguments(
        output_dir=f"{BASE_DIR}/itn_tmp",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        seed=SEED,
    )
    itn_trainer = Trainer(model=itn_model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"], tokenizer=itn_tokenizer, compute_metrics=metrics)
    itn_trainer.train()

    pred = itn_trainer.predict(ds["test"])
    y_true = pred.label_ids
    y_sup = np.argmax(pred.predictions, axis=-1)
    y_rule = [semiotic_label2id.get(tagger_heuristic.tag_span(r["span"], r["sentence"]), -1) for _, r in test_df.iterrows()]
    keep = [i for i, y in enumerate(y_rule) if y >= 0]

    print("\nMethod 1 — GIZA++/Thutmose rules:")
    print(classification_report([y_true[i] for i in keep], [y_rule[i] for i in keep], target_names=SEMIOTIC_LABELS, zero_division=0))
    print("\nMethod 2 — supervised ITN model:")
    print(classification_report(y_true, y_sup, target_names=SEMIOTIC_LABELS, zero_division=0))

    benchmark_itn = pd.DataFrame([
        {"method": "giza_thutmose_rules", "accuracy": accuracy_score([y_true[i] for i in keep], [y_rule[i] for i in keep]), "macro_f1": f1_score([y_true[i] for i in keep], [y_rule[i] for i in keep], average="macro")},
        {"method": "supervised_itn_xlm_roberta", "accuracy": accuracy_score(y_true, y_sup), "macro_f1": f1_score(y_true, y_sup, average="macro")},
    ]).sort_values("macro_f1", ascending=False)
    display(benchmark_itn)

    os.makedirs(ITN_MODEL_DIR, exist_ok=True)
    itn_trainer.save_model(ITN_MODEL_DIR)
    itn_tokenizer.save_pretrained(ITN_MODEL_DIR)
    itn_test_df = test_df

if RUN_ITN_SUPERVISED_TRAINING:
    train_itn_supervised_model()

def classify_itn_span_supervised(span, sentence):
    if "itn_model" not in globals() or "itn_tokenizer" not in globals():
        return tagger_heuristic.tag_span(span, sentence)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    itn_model.to(device).eval()
    text = f"span: {span} </s> sentence: {sentence}"
    enc = itn_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        pred = torch.argmax(itn_model(**enc).logits, dim=-1).item()
    return semiotic_id2label[pred]

class SupervisedITNTagger:
    def tag_text(self, text):
        candidates = tagger_heuristic.extract_candidates(text)
        for item in candidates:
            item["class_rules"] = item["class"]
            item["class"] = classify_itn_span_supervised(item["token"], text)
        return candidates
    def summary(self, text):
        out = {}
        for item in self.tag_text(text):
            cls = item["class"]
            if cls != "O":
                out[cls] = out.get(cls, 0) + 1
        return out

tagger_supervised = SupervisedITNTagger()
tagger = tagger_supervised if "itn_model" in globals() else tagger_heuristic
print("Semiotic tagging ready. Active method:", "supervised_itn_xlm_roberta" if "itn_model" in globals() else "giza_thutmose_rules")


## STEP 8 — Full Corpus Processing + Excel Generation

In [15]:
from tqdm import tqdm
import re
print(f'Corpus entries: {len(df_corpus)}')
cols=df_corpus.columns.tolist(); ID_COL='audio_id' if 'audio_id' in cols else cols[0]; GT_COL='ground_truth' if 'ground_truth' in cols else (cols[1] if len(cols)>1 else None); QWEN_COL=next((c for c in cols if 'qwen' in c.lower() or 'asr' in c.lower()), None)
print(f'  ID={ID_COL} | GT={GT_COL} | ASR={QWEN_COL}')
results=[]
for _,row in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc='Processing corpus'):
    aid=row[ID_COL]; gt=str(row[GT_COL]) if GT_COL and pd.notna(row.get(GT_COL,None)) else ''; qwen=str(row[QWEN_COL]) if QWEN_COL and pd.notna(row.get(QWEN_COL,None)) else ''
    spoken=re.sub(r'\s+',' ', rm_dir(gt.strip())).strip(); written=spoken_to_written(spoken, normalize_numbers=False); wn=spoken_to_written(spoken, normalize_numbers=True)
    ld_rules=lang_dist(spoken, method='rules'); ld_sup=lang_dist(spoken, method='supervised'); ltoks=analyze_lang(spoken, method='supervised')
    fren=[t['token'] for t in ltoks if t['lang'] in ['FRENCH','ENGLISH','FR','EN','MIXED']]
    aligns=align_tokens(spoken.split(), written.split(), spoken_text=spoken, written_text=written)
    astr=' | '.join(f'{a["st"]}->{a["wt"]}' for a in aligns if a.get('st') != a.get('wt') and a.get('wt'))[:200]
    try: sem=tagger.summary(written)
    except Exception as e: sem={'THUTMOSE_ERROR':1}; print('Thutmose error:', e)
    results.append({'audio_id':aid,'spoken':spoken,'written':written,'written_normalized':wn,'lang_AR_DA_pct':ld_sup.get('ARABIC',0)+ld_sup.get('DARIJA',0),'lang_FR_pct':ld_sup.get('FRENCH',0),'lang_EN_pct':ld_sup.get('ENGLISH',0),'fr_en_tokens':', '.join(fren),'alignment_pairs':astr,'semiotic_classes':', '.join(f'{k}({v})' for k,v in sem.items()) or 'PLAIN','normalization_applied':'oui' if spoken!=written else '','token_count':len(spoken.split()),'ground_truth_original':gt,'qwen_transcription':qwen,'lang_id_method':'supervised_xlm_roberta' if USE_SUPERVISED_LANG_ID else 'rules_first_pass','lang_rules_dist':str(ld_rules),'lang_supervised_dist':str(ld_sup),'alignment_method':'giza++' if GIZA_ALIGN_TABLE else 'fallback_length','itn_tagger':'supervised_itn_xlm_roberta' if 'itn_model' in globals() else 'giza_thutmose_rules'})
df_results=pd.DataFrame(results)
print(f'Done: {len(df_results)} entries')
print(f'  Lang ID method: {"supervised_xlm_roberta" if USE_SUPERVISED_LANG_ID else "rules_first_pass"}')
print(f'  Alignment method: {"giza++" if GIZA_ALIGN_TABLE else "fallback_length"}')


Corpus entries: 508
  ID=audio_id | GT=ground_truth | ASR=qwen3-asr-checkpoint700


Processing corpus:   0%|          | 1/508 [00:03<33:24,  3.95s/it]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:   1%|          | 4/508 [00:04<06:31,  1.29it/s]

Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['g

fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:   1%|▏         | 7/508 [00:04<03:10,  2.63it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:   2%|▏         | 12/508 [00:04<01:29,  5.55it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kagg

Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero 

fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:   3%|▎         | 17/508 [00:04<00:54,  9.04it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:   4%|▍         | 22/508 [00:04<00:37, 13.02it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.


... (105 outputs left out)

fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:  97%|█████████▋| 493/508 [00:15<00:00, 41.95it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:  98%|█████████▊| 498/508 [00:16<00:00, 42.24it/s]

Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero 

fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Processing corpus:  99%|█████████▉| 503/508 [00:16<00:00, 42.10it/s]fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/NeMo' already exists and is not an empty directory.
Pr

Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero exit status 128.
Thutmose error: Real Thutmose Tagger script is unavailable. Add NeMo examples or set THUTMOSE_SCRIPT manually.
Could not clone NeMo examples: Command '['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo.git', '/kaggle/working/NeMo']' returned non-zero 

In [16]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def make_excel(df, path):
    W='FFFFFF'; N='1F3864'; B='2E75B6'
    wb=openpyxl.Workbook(); ws=wb.active
    ws.title='Spoken_Written'; ws.sheet_view.rightToLeft=True
    col_keys=['audio_id','spoken','written','written_normalized','lang_AR_DA_pct','lang_FR_pct','lang_EN_pct','fr_en_tokens','alignment_pairs','semiotic_classes','normalization_applied','token_count','ground_truth_original','qwen_transcription','lang_id_method','lang_rules_dist','lang_supervised_dist','alignment_method','itn_tagger']
    hdrs=['audio_id','spoken','written','written_norm','AR/DA%','FR%','EN%','FR/EN tokens','alignments','semiotic cls','normalized?','tokens','ground_truth','qwen','lang_id_method','rules_dist','supervised_dist','alignment_method','itn_tagger']
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(hdrs))
    t=ws.cell(1,1); t.value='Darija Speech Corpus — GIZA++ / Thutmose / Benchmark Edition'
    t.font=Font(bold=True,size=14,color=W,name='Arial'); t.fill=PatternFill('solid',start_color=N); t.alignment=Alignment(horizontal='center',vertical='center')
    for ci,h in enumerate(hdrs,1):
        c=ws.cell(row=2,column=ci,value=h); c.font=Font(bold=True,color=W,name='Arial',size=10); c.fill=PatternFill('solid',start_color=B); c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
    thin=Side(style='thin',color='CCCCCC'); brd=Border(left=thin,right=thin,top=thin,bottom=thin)
    for ri,row in df.iterrows():
        fill=PatternFill('solid',start_color='EEF3FB' if ri%2==0 else 'FFFFFF')
        for ci,k in enumerate(col_keys,1):
            c=ws.cell(row=ri+3,column=ci,value=row.get(k,'')); c.fill=fill; c.border=brd; c.alignment=Alignment(horizontal='right' if ci<=4 else 'center',vertical='center',wrap_text=True)
    widths=[12,30,30,30,8,8,8,20,25,18,10,8,30,30,18,25,25,16,18]
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w
    if 'benchmark_lang_id' in globals() and isinstance(benchmark_lang_id, pd.DataFrame) and len(benchmark_lang_id):
        ws2=wb.create_sheet('Benchmark_Lang_ID')
        for ci,h in enumerate(benchmark_lang_id.columns,1): ws2.cell(row=1,column=ci,value=h).font=Font(bold=True)
        for ri,(_,row) in enumerate(benchmark_lang_id.iterrows(),2):
            for ci,h in enumerate(benchmark_lang_id.columns,1): ws2.cell(row=ri,column=ci,value=row[h])
    wb.save(path); print(f'Excel saved: {path}')

EXCEL_OUT=f'{OUTPUT_DIR}/corpus_giza_thutmose_benchmark.xlsx'
make_excel(df_results, EXCEL_OUT)


Excel saved: /kaggle/working/darija_corpus/outputs/corpus_giza_thutmose_benchmark.xlsx


## STEP 9 — Final Report & Download

In [19]:
import os, zipfile

print('='*60)
print('DARIJA PIPELINE FINAL REPORT')
print('='*60)
print(f"  Total corpus       : {len(df_results)}")
print(f"  Normalized texts   : {df_results['normalization_applied'].eq('oui').sum()}")
print(f"  With FR/EN tokens  : {df_results['fr_en_tokens'].ne('').sum()}")
print(f"  Avg tokens/text    : {df_results['token_count'].mean():.1f}")
print(f"  Lang ID method     : {'supervised_xlm_roberta' if USE_SUPERVISED_LANG_ID else 'rules_first_pass'}")
print(f"  Denoising method   : {globals().get('DENOISE_METHOD', 'unknown')}")
print(f"  NeMo model         : {globals().get('nemo_enhancement_name', 'sample_only_or_not_loaded')}")
print(f"  Alignment method   : {'GIZA++' if GIZA_ALIGN_TABLE else 'fallback_length'}")
print(f"  ITN tagger         : {'supervised_itn_xlm_roberta' if 'itn_model' in globals() else 'giza_thutmose_rules'}")

if 'benchmark_lang_id' in globals() and len(benchmark_lang_id):
    print('\nLanguage ID benchmark:')
    print(benchmark_lang_id.to_string(index=False))
if 'benchmark_itn' in globals() and len(benchmark_itn):
    print('\nSemiotic ITN benchmark:')
    print(benchmark_itn.to_string(index=False))

print('='*60)

CSV_PATH = f'{OUTPUT_DIR}/corpus_pipeline_giza_thutmose_benchmark.csv'
df_results.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f'CSV saved: {CSV_PATH}')

ZIP_OUT = '/kaggle/working/Darija_Pipeline_GIZA_Thutmose_Benchmark.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(fp):
            zf.write(fp, f)

print(f'ZIP: {ZIP_OUT} ({os.path.getsize(ZIP_OUT)/1024/1024:.2f} MB)')
print('Download: Kaggle sidebar -> Output tab -> Download')
print('Pipeline complete!')


DARIJA PIPELINE FINAL REPORT
  Total corpus       : 508
  Normalized texts   : 473
  With FR/EN tokens  : 454
  Avg tokens/text    : 33.2
  Lang ID method     : supervised_xlm_roberta
  Denoising method   : NeMo sample(10) + standardized copy for the rest
  NeMo model         : nvidia/se_den_sb_16k_small
  Alignment method   : fallback_length
  ITN tagger         : real Thutmose Tagger through NeMo

Language ID benchmark:
                method  accuracy  macro_f1  weighted_f1
supervised_xlm_roberta  0.696501  0.687292     0.687520
      rules_first_pass  0.270525  0.133693     0.136596
CSV saved: /kaggle/working/darija_corpus/outputs/corpus_pipeline_giza_thutmose_benchmark.csv
ZIP: /kaggle/working/Darija_Pipeline_GIZA_Thutmose_Benchmark.zip (0.37 MB)
Download: Kaggle sidebar -> Output tab -> Download
Pipeline complete!


## Notes

### Added in this version

- Audio data augmentation is enabled with controlled growth: gain, noise, time shift, speed, pitch, polarity, clipping, band-pass and reverb.
- UTF-8 Arabic normalization mappings are commented directly in the code.
- Semiotic labels are defined with two methods:
  - **Method 1:** GIZA++ alignment plus a complete Thutmose-style rule tagger.
  - **Method 2:** supervised learning from `itn_annotated_all.parquet` using `sentence`, `span`, `label`.
- The semiotic classes are: `CARDINAL`, `MONEY`, `DATE`, `ORDINAL`, `MEASURE`, `DECIMAL`, `LETTER`, `TIME`, `PUNCT`, `TELEPHONE`, `ADDRESS`, `ELECTRONIC`.
- A benchmark compares Method 1 and Method 2 using accuracy and macro F1.
